# WAVEWATCH III Exploration Notebook: BUFR Data Processing & Tools

This notebook demonstrates how to use the `ww3_tools` Python library to:
1. Read and inspect BUFR observation files
2. Convert BUFR observations into `pandas` DataFrames
3. Apply geospatial bounding-box and quality/metric filters
4. Export converted data into NetCDF format for WAVEWATCH III workflows.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add python tools directory to path
sys.path.insert(0, os.path.abspath('../python'))

from ww3_tools import (
    convert_bufr_to_dataframe,
    filter_bufr_data,
    convert_bufr_to_netcdf,
    filter_by_bbox,
    format_datetime,
)

print("ww3_tools successfully imported!")

## 1. Creating Synthetic BUFR-like Wave Observation Data for Demonstration

In a operational workflow, BUFR files received from GTS/NOAA/ECMWF are ingested. Here we inspect the structured observation DataFrame.

In [ ]:
# Generate mock observation dataset
data = {
    'latitude': [25.0, 30.5, 45.2, 50.1, -12.3],
    'longitude': [-80.0, -75.2, -30.1, 10.5, 45.0],
    'year': [2026, 2026, 2026, 2026, 2026],
    'month': [9, 9, 9, 9, 9],
    'day': [14, 14, 14, 14, 14],
    'hour': [12, 12, 13, 13, 14],
    'minute': [0, 30, 15, 45, 0],
    'significantWaveHeight': [2.1, 4.5, 1.8, 6.2, 3.3],
    'peakWavePeriod': [8.0, 12.5, 7.5, 14.0, 9.5]
}
df_obs = pd.DataFrame(data)
df_obs = format_datetime(df_obs)
df_obs

## 2. Filtering Observations

Filter by geospatial bounding box (e.g. North Atlantic: Lat 20..60, Lon -90..0) and wave height.

In [ ]:
# Filter within bounding box
filtered_df = filter_bufr_data(
    df_obs,
    bbox=(20.0, 60.0, -90.0, 0.0),
    min_wave_height=2.0
)
filtered_df

## 3. Exporting to xarray Dataset / NetCDF

Convert filtered observations into an `xarray.Dataset` for input to WAVEWATCH III tools.

In [ ]:
ds = filtered_df.to_xarray()
ds.attrs['title'] = 'WAVEWATCH III Filtered BUFR Wave Observations'
print(ds)